# 02 — Binance WebSocket API Exploration
Phase 1: Understanding the live streaming data structure and how it differs from the REST API.

## 1. Setup

In [2]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import asyncio
import websockets
import json
import pandas as pd

from src.params.constants import BINANCE_KLINES_COLUMNS, BINANCE_WS_BASE_URL, DEFAULT_WS_INTERVAL
from src.params.enums import Symbol

## 2. Connect and Collect Raw Messages
Connect to the Binance kline stream for BTCUSDT at 1-minute interval and collect 5 raw tick messages.

In [3]:
WS_URL = f"{BINANCE_WS_BASE_URL}/{Symbol.BTCUSDT.value.lower()}@kline_{DEFAULT_WS_INTERVAL.value}"

async def collect_ticks(n=5):
    messages = []
    async with websockets.connect(WS_URL) as ws:
        print(f"Connected to: {WS_URL}")
        print(f"Collecting {n} tick messages...\n")
        for i in range(n):
            msg = await ws.recv()
            messages.append(json.loads(msg))
            print(f"  Received tick {i + 1}/{n}")
    return messages

ticks = await collect_ticks(5)
print("\nDone.")

Connected to: wss://stream.binance.com:9443/ws/btcusdt@kline_1m

  Received tick 1/5
  Received tick 2/5
  Received tick 3/5
  Received tick 4/5
  Received tick 5/5

Done.


## 3. Inspect Raw Message Structure
Print the first raw message exactly as Binance sends it.

In [4]:
print("=== Full raw WebSocket message ===")
print(json.dumps(ticks[0], indent=2))

=== Full raw WebSocket message ===
{
  "e": "kline",
  "E": 1778679830015,
  "s": "BTCUSDT",
  "k": {
    "t": 1778679780000,
    "T": 1778679839999,
    "s": "BTCUSDT",
    "i": "1m",
    "f": 6292092767,
    "L": 6292096958,
    "o": "79934.01000000",
    "c": "79882.80000000",
    "h": "79954.78000000",
    "l": "79869.94000000",
    "v": "18.04347000",
    "n": 4192,
    "x": false,
    "q": "1441673.00760160",
    "V": "10.34232000",
    "Q": "826254.35663910",
    "B": "0"
  }
}


## 4. Understand the Message Structure
A WebSocket message has two layers — the outer event envelope and the inner kline payload.

In [5]:
msg = ticks[0]

print("=== Outer envelope fields ===")
for key, value in msg.items():
    if key != "k":
        print(f"  {key!r:6} = {value}")

print("\n=== Inner kline payload (msg['k']) ===")
for key, value in msg["k"].items():
    print(f"  {key!r:6} = {value}")

=== Outer envelope fields ===
  'e'    = kline
  'E'    = 1778679830015
  's'    = BTCUSDT

=== Inner kline payload (msg['k']) ===
  't'    = 1778679780000
  'T'    = 1778679839999
  's'    = BTCUSDT
  'i'    = 1m
  'f'    = 6292092767
  'L'    = 6292096958
  'o'    = 79934.01000000
  'c'    = 79882.80000000
  'h'    = 79954.78000000
  'l'    = 79869.94000000
  'v'    = 18.04347000
  'n'    = 4192
  'x'    = False
  'q'    = 1441673.00760160
  'V'    = 10.34232000
  'Q'    = 826254.35663910
  'B'    = 0


## 5. Decode the Field Names
Binance uses single-letter keys in WebSocket to reduce payload size. Map them to readable names.

In [6]:
ws_field_map = {
    # Outer envelope
    "e": "event_type",
    "E": "event_time",
    "s": "symbol",
    # Inner kline payload
    "t": "kline_start_time",
    "T": "kline_close_time",
    "i": "interval",
    "o": "open",
    "c": "close",
    "h": "high",
    "l": "low",
    "v": "volume",
    "n": "number_of_trades",
    "x": "is_candle_closed",
    "q": "quote_asset_volume",
    "V": "taker_buy_base_volume",
    "Q": "taker_buy_quote_volume",
    "f": "first_trade_id",
    "L": "last_trade_id",
    "B": "ignore",
}

for short, full in ws_field_map.items():
    print(f"  {short!r:6} => {full}")

  'e'    => event_type
  'E'    => event_time
  's'    => symbol
  't'    => kline_start_time
  'T'    => kline_close_time
  'i'    => interval
  'o'    => open
  'c'    => close
  'h'    => high
  'l'    => low
  'v'    => volume
  'n'    => number_of_trades
  'x'    => is_candle_closed
  'q'    => quote_asset_volume
  'V'    => taker_buy_base_volume
  'Q'    => taker_buy_quote_volume
  'f'    => first_trade_id
  'L'    => last_trade_id
  'B'    => ignore


## 6. The `is_candle_closed` Field — Why It Matters
Binance sends a tick every second even while the candle is still forming.
`is_candle_closed = False` means the candle is still updating.
`is_candle_closed = True` means the candle is final and will never change.

For MongoDB: you can store all ticks (for real-time dashboard) or only closed candles (for ML).

In [7]:
print("=== is_candle_closed status across collected ticks ===")
for i, tick in enumerate(ticks):
    symbol = tick["s"]
    close_price = tick["k"]["c"]
    is_closed = tick["k"]["x"]
    print(f"  Tick {i + 1}: symbol={symbol}  close={close_price}  is_candle_closed={is_closed}")

=== is_candle_closed status across collected ticks ===
  Tick 1: symbol=BTCUSDT  close=79882.80000000  is_candle_closed=False
  Tick 2: symbol=BTCUSDT  close=79886.95000000  is_candle_closed=False
  Tick 3: symbol=BTCUSDT  close=79879.69000000  is_candle_closed=False
  Tick 4: symbol=BTCUSDT  close=79884.79000000  is_candle_closed=False
  Tick 5: symbol=BTCUSDT  close=79866.82000000  is_candle_closed=False


## 7. Parse a Tick into a Clean Document
This is what one MongoDB document will look like — flat structure, no deep nesting.

In [8]:
def parse_tick(msg):
    k = msg["k"]
    return {
        "symbol":           msg["s"],
        "event_time":       pd.to_datetime(msg["E"], unit="ms"),
        "kline_start_time": pd.to_datetime(k["t"], unit="ms"),
        "kline_close_time": pd.to_datetime(k["T"], unit="ms"),
        "interval":         k["i"],
        "open":             float(k["o"]),
        "high":             float(k["h"]),
        "low":              float(k["l"]),
        "close":            float(k["c"]),
        "volume":           float(k["v"]),
        "number_of_trades": int(k["n"]),
        "is_candle_closed": bool(k["x"]),
    }

parsed = parse_tick(ticks[0])
for field, value in parsed.items():
    print(f"  {field:<20} {value}")

  symbol               BTCUSDT
  event_time           2026-05-13 13:43:50.015000
  kline_start_time     2026-05-13 13:43:00
  kline_close_time     2026-05-13 13:43:59.999000
  interval             1m
  open                 79934.01
  high                 79954.78
  low                  79869.94
  close                79882.8
  volume               18.04347
  number_of_trades     4192
  is_candle_closed     False


## 8. REST API vs WebSocket — Field Comparison
This comparison directly justifies why we need two separate database schemas.

In [9]:
rest_fields = set(BINANCE_KLINES_COLUMNS)
ws_fields   = set(parse_tick(ticks[0]).keys())

print("=== Fields in REST API only ===")
for f in sorted(rest_fields - ws_fields):
    print(f"  {f}")

print("\n=== Fields in WebSocket only ===")
for f in sorted(ws_fields - rest_fields):
    print(f"  {f}")

print("\n=== Fields shared by both ===")
for f in sorted(rest_fields & ws_fields):
    print(f"  {f}")

=== Fields in REST API only ===
  close_time
  ignore
  open_time
  quote_asset_volume
  taker_buy_base_volume
  taker_buy_quote_volume

=== Fields in WebSocket only ===
  event_time
  interval
  is_candle_closed
  kline_close_time
  kline_start_time
  symbol

=== Fields shared by both ===
  close
  high
  low
  number_of_trades
  open
  volume


## 9. Summary — Architecture Decision
Write your findings here before filling in `references/phase1/schema_design.md`.

In [10]:
summary = """
REST API (historical):
  - Returns a list of 12 fixed fields per candle
  - All candles are closed
  - Best stored in PostgreSQL: fixed schema, queried for ML training

WebSocket (streaming):
  - Returns JSON events with nested kline payload
  - Uses single-letter keys (compressed for speed)
  - Sends updates while candle is still forming (is_candle_closed=False)
  - Has extra fields: event_time, interval, is_candle_closed
  - Best stored in MongoDB: flexible JSON, high write rate, schema may evolve

Conclusion:
  Two different sources → two different structures → two different databases.
  PostgreSQL for historical OHLCV, MongoDB for live ticks.
"""
print(summary)


REST API (historical):
  - Returns a list of 12 fixed fields per candle
  - All candles are closed
  - Best stored in PostgreSQL: fixed schema, queried for ML training

WebSocket (streaming):
  - Returns JSON events with nested kline payload
  - Uses single-letter keys (compressed for speed)
  - Sends updates while candle is still forming (is_candle_closed=False)
  - Has extra fields: event_time, interval, is_candle_closed
  - Best stored in MongoDB: flexible JSON, high write rate, schema may evolve

Conclusion:
  Two different sources → two different structures → two different databases.
  PostgreSQL for historical OHLCV, MongoDB for live ticks.

